# 글로벌 Peer Group 분석 및 시각화

## 목적
- LG전자와 글로벌 경쟁사들의 재무성과 비교 분석
- 다양한 시각화를 통한 경쟁 포지션 파악
- 투자 지표 및 수익성 분석

## 분석 대상 기업
- **LG전자** (한국)
- **Whirlpool** (미국) 
- **Electrolux** (스웨덴)
- **Haier** (중국)
- **Daikin** (일본)
- **Hisense** (중국)
- **Continental** (독일)
- **Denso** (일본)

In [3]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 그래프 스타일 설정
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📊 글로벌 Peer Group 분석을 시작합니다!")
print("=" * 50)

📊 글로벌 Peer Group 분석을 시작합니다!


In [4]:
# 글로벌 Peer Group 데이터 입력
companies_data = {
    '회사명': ['LG전자', 'Whirlpool', 'Electrolux', 'Haier', 'Daikin', 'Hisense', 'Continental', 'Denso'],
    '티커': ['066570 KS', 'WHR US', 'ELUXB SS', '600690 CH', '6367 JT', '600060 CH', 'CON GR', '6902 JT'],
    '국가': ['한국', '미국', '스웨덴', '중국', '일본', '중국', '독일', '일본'],
    '주가_로컬': [97200, 99.3, 101.2, 31.3, 18085.0, 18.8, 60.2, 2112.0],
    '시가총액_USD_mn': [15906.6, 5449.9, 2717.3, 39142.8, 34851.8, 3447.4, 12998.8, 43764.4],
    
    # 매출액 YoY (%)
    '매출액_YoY_23': [0.9, -1.4, -0.3, 7.9, 28.1, 17.8, 5.1, 16.1],
    '매출액_YoY_24F': [5.6, -13.9, -0.9, 6.9, 15.7, 12.0, 0.1, 13.7],
    '매출액_YoY_25F': [5.3, -1.9, 3.0, 7.2, 4.4, 11.1, 3.9, 5.8],
    
    # 영업이익 YoY (%)
    '영업이익_YoY_23': [-0.1, -6.7, -152.2, 23.0, 19.2, 47.5, -0.1, 20.7],
    '영업이익_YoY_24F': [20.6, -6.3, 215.2, 13.8, 13.1, 6.5, 32.0, 47.3],
    '영업이익_YoY_25F': [9.8, 15.1, 174.0, 12.8, 9.8, 16.2, 22.8, 23.5],
    
    # 영업이익률 (%)
    '영업이익률_23': [4.2, 5.8, 0.5, 7.5, 9.5, 5.5, 4.6, 6.7],
    '영업이익률_24F': [4.8, 6.4, 1.5, 8.0, 9.3, 5.2, 6.1, 8.7],
    '영업이익률_25F': [5.0, 7.5, 4.1, 8.4, 9.7, 5.5, 7.2, 10.1],
    
    # P/E (x)
    'PE_23': [25.8, 11.6, np.nan, 11.7, 23.2, 12.9, 13.3, 27.5],
    'PE_24F': [9.2, 8.6, np.nan, 15.4, 20.5, 11.3, 8.3, 13.1],
    'PE_25F': [6.9, 7.8, 9.9, 13.7, 16.9, 9.9, 6.3, 11.1],
    
    # P/B (x)
    'PB_23': [0.8, 2.8, 2.6, 1.9, 2.3, 1.4, 1.1, 1.5],
    'PB_24F': [0.8, 1.7, 2.5, 2.6, 1.9, 1.2, 0.8, 1.1],
    'PB_25F': [0.7, 1.7, 2.0, 2.3, 1.8, 1.1, 0.8, 1.0],
    
    # EV/EBITDA (x)
    'EV_EBITDA_23': [3.9, 8.2, 16.6, 6.8, 10.4, 8.5, 5.4, 11.5],
    'EV_EBITDA_24F': [3.3, 8.3, 6.6, 8.7, 8.4, 10.3, 3.5, 6.2],
    'EV_EBITDA_25F': [2.9, 7.1, 4.3, 7.6, 7.3, 9.9, 3.0, 5.8],
    
    # ROE (%)
    'ROE_23': [3.3, 20.5, -37.7, 16.9, 10.7, 11.5, 8.6, 6.3],
    'ROE_24F': [8.8, 12.0, -2.4, 16.8, 10.1, 10.5, 9.8, 8.4],
    'ROE_25F': [10.7, 20.7, 22.7, 16.9, 10.2, 11.2, 12.0, 10.1],
    
    # EPS (로컬)
    'EPS_23': [3954, 8.8, -19.4, 1.8, 889.2, 1.6, 5.8, 105.0],
    'EPS_24F': [10596, 11.5, -0.9, 2.0, 881.8, 1.7, 7.2, 161.4],
    'EPS_25F': [14105, 12.7, 10.2, 2.3, 1068.3, 1.9, 9.5, 191.0]
}

# DataFrame 생성
df = pd.DataFrame(companies_data)

print("📋 데이터 로딩 완료!")
print(f"   • 분석 대상 기업: {len(df)}개")
print(f"   • 데이터 컬럼: {len(df.columns)}개")
print("\n🏢 분석 대상 기업 목록:")
for i, (company, country) in enumerate(zip(df['회사명'], df['국가']), 1):
    print(f"   {i}. {company} ({country})")

# 기본 데이터 확인
print(f"\n📊 데이터 미리보기:")
display(df.head())

📋 데이터 로딩 완료!
   • 분석 대상 기업: 8개
   • 데이터 컬럼: 29개

🏢 분석 대상 기업 목록:
   1. LG전자 (한국)
   2. Whirlpool (미국)
   3. Electrolux (스웨덴)
   4. Haier (중국)
   5. Daikin (일본)
   6. Hisense (중국)
   7. Continental (독일)
   8. Denso (일본)

📊 데이터 미리보기:


,회사명,티커,국가,주가_로컬,시가총액_USD_mn,매출액_YoY_23,매출액_YoY_24F,매출액_YoY_25F,영업이익_YoY_23,영업이익_YoY_24F,...,PB_25F,EV_EBITDA_23,EV_EBITDA_24F,EV_EBITDA_25F,ROE_23,ROE_24F,ROE_25F,EPS_23,EPS_24F,EPS_25F
0,LG전자,066570 KS,한국,97200.0,15906.6,0.9,5.6,5.3,-0.1,20.6,...,0.7,3.9,3.3,2.9,3.3,8.8,10.7,3954.0,10596.0,14105.0
1,Whirlpool,WHR US,미국,99.3,5449.9,-1.4,-13.9,-1.9,-6.7,-6.3,...,1.7,8.2,8.3,7.1,20.5,12.0,20.7,8.8,11.5,12.7
2,Electrolux,ELUXB SS,스웨덴,101.2,2717.3,-0.3,-0.9,3.0,-152.2,215.2,...,2.0,16.6,6.6,4.3,-37.7,-2.4,22.7,-19.4,-0.9,10.2
3,Haier,600690 CH,중국,31.3,39142.8,7.9,6.9,7.2,23.0,13.8,...,2.3,6.8,8.7,7.6,16.9,16.8,16.9,1.8,2.0,2.3
4,Daikin,6367 JT,일본,18085.0,34851.8,28.1,15.7,4.4,19.2,13.1,...,1.8,10.4,8.4,7.3,10.7,10.1,10.2,889.2,881.8,1068.3


In [5]:
# 1. 시가총액 비교 시각화
print("📈 1. 시가총액 비교 분석")
print("-" * 40)

# 시가총액 데이터 정렬
market_cap_data = df[['회사명', '국가', '시가총액_USD_mn']].sort_values('시가총액_USD_mn', ascending=False)

# LG전자 하이라이트 색상 설정
colors = ['#e74c3c' if company == 'LG전자' else '#3498db' for company in market_cap_data['회사명']]

# Plotly 바차트 생성
fig1 = go.Figure(data=[
    go.Bar(
        x=market_cap_data['회사명'],
        y=market_cap_data['시가총액_USD_mn'],
        text=market_cap_data['시가총액_USD_mn'].apply(lambda x: f'${x:,.0f}M'),
        textposition='auto',
        marker_color=colors,
        hovertemplate='<b>%{x}</b><br>시가총액: $%{y:,.0f}M<extra></extra>'
    )
])

fig1.update_layout(
    title={
        'text': '🏢 글로벌 Peer Group 시가총액 비교',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial Black'}
    },
    xaxis_title='회사명',
    yaxis_title='시가총액 (USD Million)',
    plot_bgcolor='white',
    height=500,
    showlegend=False
)

fig1.update_xaxes(tickangle=45)
fig1.show()

# 순위 출력
print("\n🏆 시가총액 순위:")
for i, (_, row) in enumerate(market_cap_data.iterrows(), 1):
    symbol = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}위"
    highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
    print(f"   {symbol} {row['회사명']} ({row['국가']}): ${row['시가총액_USD_mn']:,.0f}M{highlight}")

# LG전자 포지션 분석
lg_rank = market_cap_data.reset_index(drop=True)[market_cap_data.reset_index(drop=True)['회사명'] == 'LG전자'].index[0] + 1
lg_market_cap = df[df['회사명'] == 'LG전자']['시가총액_USD_mn'].iloc[0]
total_companies = len(df)

print(f"\n📊 LG전자 시가총액 분석:")
print(f"   • 순위: {lg_rank}/{total_companies}위")
print(f"   • 시가총액: ${lg_market_cap:,.0f}M")
print(f"   • 상위 비율: {((total_companies - lg_rank + 1) / total_companies * 100):.1f}%")

📈 1. 시가총액 비교 분석
----------------------------------------



🏆 시가총액 순위:
   🥇 Denso (일본): $43,764M
   🥈 Haier (중국): $39,143M
   🥉 Daikin (일본): $34,852M
   4위 LG전자 (한국): $15,907M ⭐
   5위 Continental (독일): $12,999M
   6위 Whirlpool (미국): $5,450M
   7위 Hisense (중국): $3,447M
   8위 Electrolux (스웨덴): $2,717M

📊 LG전자 시가총액 분석:
   • 순위: 4/8위
   • 시가총액: $15,907M
   • 상위 비율: 62.5%


In [6]:
# 2. 영업이익률 추이 및 비교 시각화
print("\n📈 2. 영업이익률 추이 분석")
print("-" * 40)

# 영업이익률 데이터 준비
margin_data = df[['회사명', '영업이익률_23', '영업이익률_24F', '영업이익률_25F']].copy()
margin_melted = pd.melt(margin_data, id_vars=['회사명'], 
                       value_vars=['영업이익률_23', '영업이익률_24F', '영업이익률_25F'],
                       var_name='연도', value_name='영업이익률')

# 연도 라벨 정리
margin_melted['연도'] = margin_melted['연도'].map({
    '영업이익률_23': '2023',
    '영업이익률_24F': '2024F',
    '영업이익률_25F': '2025F'
})

# Plotly 라인 차트 생성
fig2 = go.Figure()

for company in df['회사명']:
    company_data = margin_melted[margin_melted['회사명'] == company]
    
    # LG전자 강조
    line_width = 4 if company == 'LG전자' else 2
    line_color = '#e74c3c' if company == 'LG전자' else None
    
    fig2.add_trace(go.Scatter(
        x=company_data['연도'],
        y=company_data['영업이익률'],
        mode='lines+markers',
        name=company,
        line=dict(width=line_width, color=line_color),
        marker=dict(size=8 if company == 'LG전자' else 6),
        hovertemplate=f'<b>{company}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>'
    ))

fig2.update_layout(
    title={
        'text': '📊 영업이익률 추이 비교 (2023-2025F)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial Black'}
    },
    xaxis_title='연도',
    yaxis_title='영업이익률 (%)',
    plot_bgcolor='white',
    height=500,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.01
    )
)

fig2.show()

# 2025F 영업이익률 순위
margin_2025 = df[['회사명', '영업이익률_25F']].sort_values('영업이익률_25F', ascending=False)

print("\n🏆 2025F 영업이익률 순위:")
for i, (_, row) in enumerate(margin_2025.iterrows(), 1):
    symbol = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}위"
    highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
    print(f"   {symbol} {row['회사명']}: {row['영업이익률_25F']:.1f}%{highlight}")

# LG전자 영업이익률 개선 분석
lg_margin_23 = df[df['회사명'] == 'LG전자']['영업이익률_23'].iloc[0]
lg_margin_25f = df[df['회사명'] == 'LG전자']['영업이익률_25F'].iloc[0]
margin_improvement = lg_margin_25f - lg_margin_23

print(f"\n📊 LG전자 영업이익률 개선 분석:")
print(f"   • 2023: {lg_margin_23:.1f}%")
print(f"   • 2025F: {lg_margin_25f:.1f}%")
print(f"   • 개선폭: {margin_improvement:+.1f}%p")
print(f"   • 개선률: {(margin_improvement/lg_margin_23*100):+.1f}%")


📈 2. 영업이익률 추이 분석
----------------------------------------



🏆 2025F 영업이익률 순위:
   🥇 Denso: 10.1%
   🥈 Daikin: 9.7%
   🥉 Haier: 8.4%
   4위 Whirlpool: 7.5%
   5위 Continental: 7.2%
   6위 Hisense: 5.5%
   7위 LG전자: 5.0% ⭐
   8위 Electrolux: 4.1%

📊 LG전자 영업이익률 개선 분석:
   • 2023: 4.2%
   • 2025F: 5.0%
   • 개선폭: +0.8%p
   • 개선률: +19.0%


In [7]:
# 3. 밸류에이션 지표 비교 (P/E, P/B, EV/EBITDA)
print("\n📈 3. 밸류에이션 지표 비교 분석")
print("-" * 40)

# 2025F 기준 밸류에이션 데이터
valuation_data = df[['회사명', 'PE_25F', 'PB_25F', 'EV_EBITDA_25F']].copy()

# 서브플롯 생성
fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=('P/E Ratio (2025F)', 'P/B Ratio (2025F)', 'EV/EBITDA (2025F)', '종합 밸류에이션 비교'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"type": "xy"}]]
)

# LG전자 하이라이트 색상
colors = ['#e74c3c' if company == 'LG전자' else '#3498db' for company in df['회사명']]

# P/E Ratio
fig3.add_trace(
    go.Bar(x=df['회사명'], y=df['PE_25F'], name='P/E', marker_color=colors,
           text=df['PE_25F'].apply(lambda x: f'{x:.1f}x' if pd.notna(x) else 'N/A'),
           textposition='auto'),
    row=1, col=1
)

# P/B Ratio
fig3.add_trace(
    go.Bar(x=df['회사명'], y=df['PB_25F'], name='P/B', marker_color=colors,
           text=df['PB_25F'].apply(lambda x: f'{x:.1f}x'),
           textposition='auto'),
    row=1, col=2
)

# EV/EBITDA
fig3.add_trace(
    go.Bar(x=df['회사명'], y=df['EV_EBITDA_25F'], name='EV/EBITDA', marker_color=colors,
           text=df['EV_EBITDA_25F'].apply(lambda x: f'{x:.1f}x'),
           textposition='auto'),
    row=2, col=1
)

# 종합 비교 (버블 차트)
fig3.add_trace(
    go.Scatter(
        x=df['PE_25F'], 
        y=df['PB_25F'],
        mode='markers+text',
        text=df['회사명'],
        textposition="top center",
        marker=dict(
            size=df['EV_EBITDA_25F']*3,  # EV/EBITDA 크기로 버블 사이즈
            color=colors,
            opacity=0.7,
            line=dict(width=2, color='DarkSlateGrey')
        ),
        name='종합',
        hovertemplate='<b>%{text}</b><br>P/E: %{x:.1f}x<br>P/B: %{y:.1f}x<br>EV/EBITDA: %{marker.size:.1f}x<extra></extra>'
    ),
    row=2, col=2
)

fig3.update_layout(
    title={
        'text': '💰 밸류에이션 지표 종합 비교 (2025F)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'family': 'Arial Black'}
    },
    height=800,
    showlegend=False
)

# 각 서브플롯 축 업데이트
fig3.update_xaxes(tickangle=45, row=1, col=1)
fig3.update_xaxes(tickangle=45, row=1, col=2)
fig3.update_xaxes(tickangle=45, row=2, col=1)
fig3.update_xaxes(title_text="P/E Ratio", row=2, col=2)
fig3.update_yaxes(title_text="P/B Ratio", row=2, col=2)

fig3.show()

# 밸류에이션 순위 분석
print("\n🏆 밸류에이션 지표 순위 (2025F, 낮을수록 저평가):")

# P/E 순위
pe_rank = df[['회사명', 'PE_25F']].dropna().sort_values('PE_25F')
print("\n📊 P/E Ratio 순위:")
for i, (_, row) in enumerate(pe_rank.iterrows(), 1):
    highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
    print(f"   {i}. {row['회사명']}: {row['PE_25F']:.1f}x{highlight}")

# P/B 순위
pb_rank = df[['회사명', 'PB_25F']].sort_values('PB_25F')
print("\n📊 P/B Ratio 순위:")
for i, (_, row) in enumerate(pb_rank.iterrows(), 1):
    highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
    print(f"   {i}. {row['회사명']}: {row['PB_25F']:.1f}x{highlight}")

# LG전자 밸류에이션 분석
lg_pe = df[df['회사명'] == 'LG전자']['PE_25F'].iloc[0]
lg_pb = df[df['회사명'] == 'LG전자']['PB_25F'].iloc[0]
lg_ev_ebitda = df[df['회사명'] == 'LG전자']['EV_EBITDA_25F'].iloc[0]

print(f"\n💡 LG전자 밸류에이션 종합 평가:")
print(f"   • P/E Ratio: {lg_pe:.1f}x (업계 평균 대비 {'저평가' if lg_pe < df['PE_25F'].mean() else '고평가'})")
print(f"   • P/B Ratio: {lg_pb:.1f}x (업계 평균 대비 {'저평가' if lg_pb < df['PB_25F'].mean() else '고평가'})")
print(f"   • EV/EBITDA: {lg_ev_ebitda:.1f}x (업계 평균 대비 {'저평가' if lg_ev_ebitda < df['EV_EBITDA_25F'].mean() else '고평가'})")


📈 3. 밸류에이션 지표 비교 분석
----------------------------------------



🏆 밸류에이션 지표 순위 (2025F, 낮을수록 저평가):

📊 P/E Ratio 순위:
   1. Continental: 6.3x
   2. LG전자: 6.9x ⭐
   3. Whirlpool: 7.8x
   4. Electrolux: 9.9x
   5. Hisense: 9.9x
   6. Denso: 11.1x
   7. Haier: 13.7x
   8. Daikin: 16.9x

📊 P/B Ratio 순위:
   1. LG전자: 0.7x ⭐
   2. Continental: 0.8x
   3. Denso: 1.0x
   4. Hisense: 1.1x
   5. Whirlpool: 1.7x
   6. Daikin: 1.8x
   7. Electrolux: 2.0x
   8. Haier: 2.3x

💡 LG전자 밸류에이션 종합 평가:
   • P/E Ratio: 6.9x (업계 평균 대비 저평가)
   • P/B Ratio: 0.7x (업계 평균 대비 저평가)
   • EV/EBITDA: 2.9x (업계 평균 대비 저평가)


In [10]:
# 4. ROE 및 성장률 분석
print("\n📈 4. ROE 및 성장률 분석")
print("-" * 40)

# ROE 추이 데이터 준비
roe_data = df[['회사명', 'ROE_23', 'ROE_24F', 'ROE_25F']].copy()
roe_melted = pd.melt(roe_data, id_vars=['회사명'], 
                    value_vars=['ROE_23', 'ROE_24F', 'ROE_25F'],
                    var_name='연도', value_name='ROE')

roe_melted['연도'] = roe_melted['연도'].map({
    'ROE_23': '2023',
    'ROE_24F': '2024F',
    'ROE_25F': '2025F'
})

# 매출액 성장률 vs ROE 산점도 (2025F 기준)
fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('ROE 추이 (2023-2025F)', '매출액 성장률 vs ROE '),
    specs=[[{"secondary_y": False}, {"secondary_y": False}]]
)

# ROE 추이 라인 차트
for company in df['회사명']:
    company_data = roe_melted[roe_melted['회사명'] == company]
    
    line_width = 4 if company == 'LG전자' else 2
    line_color = '#d8e614' if company == 'LG전자' else None
    
    fig4.add_trace(
        go.Scatter(
            x=company_data['연도'],
            y=company_data['ROE'],
            mode='lines+markers',
            name=company,
            line=dict(width=line_width, color=line_color),
            marker=dict(size=8 if company == 'LG전자' else 6),
            legendgroup=company,
            showlegend=True
        ),
        row=1, col=1
    )

# 매출액 성장률 vs ROE 산점도
colors_scatter = ["#d8e614" if company == 'LG전자' else "#7c9eb4" for company in df['회사명']]

fig4.add_trace(
    go.Scatter(
        x=df['매출액_YoY_25F'],
        y=df['ROE_25F'],
        mode='markers+text',
        text=df['회사명'],
        textposition="top center",
        marker=dict(
            size=12,
            color=colors_scatter,
            opacity=0.7,
            line=dict(width=2, color='DarkSlateGrey')
        ),
        name='Growth vs ROE',
        showlegend=False,
        hovertemplate='<b>%{text}</b><br>매출 성장률: %{x:.1f}%<br>ROE: %{y:.1f}%<extra></extra>'
    ),
    row=1, col=2
)

fig4.update_layout(
    title={
        'text': '📊 수익성 및 성장성 분석',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'family': 'Arial Black'}
    },
    height=500,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.01
    )
)

fig4.update_xaxes(title_text="매출액 성장률 (%)", row=1, col=2)
fig4.update_yaxes(title_text="ROE (%)", row=1, col=1)
fig4.update_yaxes(title_text="ROE (%)", row=1, col=2)

fig4.show()

# ROE 순위 (2025F)
roe_2025 = df[['회사명', 'ROE_25F']].sort_values('ROE_25F', ascending=False)

print("\n🏆 2025F ROE 순위:")
for i, (_, row) in enumerate(roe_2025.iterrows(), 1):
    symbol = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}위"
    highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
    print(f"   {symbol} {row['회사명']}: {row['ROE_25F']:.1f}%{highlight}")

# LG전자 ROE 개선 분석
lg_roe_23 = df[df['회사명'] == 'LG전자']['ROE_23'].iloc[0]
lg_roe_25f = df[df['회사명'] == 'LG전자']['ROE_25F'].iloc[0]
roe_improvement = lg_roe_25f - lg_roe_23

print(f"\n📊 LG전자 ROE 개선 분석:")
print(f"   • 2023: {lg_roe_23:.1f}%")
print(f"   • 2025F: {lg_roe_25f:.1f}%")
print(f"   • 개선폭: {roe_improvement:+.1f}%p")
print(f"   • 개선률: {(roe_improvement/abs(lg_roe_23)*100):+.1f}%")


📈 4. ROE 및 성장률 분석
----------------------------------------



🏆 2025F ROE 순위:
   🥇 Electrolux: 22.7%
   🥈 Whirlpool: 20.7%
   🥉 Haier: 16.9%
   4위 Continental: 12.0%
   5위 Hisense: 11.2%
   6위 LG전자: 10.7% ⭐
   7위 Daikin: 10.2%
   8위 Denso: 10.1%

📊 LG전자 ROE 개선 분석:
   • 2023: 3.3%
   • 2025F: 10.7%
   • 개선폭: +7.4%p
   • 개선률: +224.2%


In [9]:
# 5. 종합 경쟁력 레이더 차트 및 최종 분석
print("\n📈 5. 종합 경쟁력 분석")
print("-" * 40)

# 주요 지표들을 정규화 (0-1 스케일)
from sklearn.preprocessing import MinMaxScaler

# 분석할 지표들 선택 (2025F 기준)
radar_metrics = ['시가총액_USD_mn', '영업이익률_25F', 'ROE_25F', '매출액_YoY_25F']
radar_labels = ['시가총액', '영업이익률', 'ROE', '매출 성장률']

# 정규화를 위한 데이터 준비
radar_data = df[['회사명'] + radar_metrics].copy()

# P/E, P/B, EV/EBITDA는 낮을수록 좋으므로 역수 사용
radar_data['PE_inv_25F'] = 1 / df['PE_25F'].fillna(df['PE_25F'].median())
radar_data['PB_inv_25F'] = 1 / df['PB_25F']
radar_data['EV_EBITDA_inv_25F'] = 1 / df['EV_EBITDA_25F']

radar_metrics_extended = radar_metrics + ['PE_inv_25F', 'PB_inv_25F', 'EV_EBITDA_inv_25F']
radar_labels_extended = radar_labels + ['밸류에이션(P/E)', '밸류에이션(P/B)', '밸류에이션(EV/EBITDA)']

# MinMax 정규화
scaler = MinMaxScaler()
normalized_data = scaler.fit_transform(radar_data[radar_metrics_extended])
normalized_df = pd.DataFrame(normalized_data, columns=radar_metrics_extended)
normalized_df['회사명'] = radar_data['회사명'].values

# 레이더 차트 생성 (주요 3개 회사 비교: LG전자, 가장 큰 시가총액, 가장 높은 ROE)
top_companies = ['LG전자', 'Denso', 'Whirlpool']  # 대표적인 3개사
colors_radar = ['#e74c3c', '#2ecc71', '#3498db']

fig5 = go.Figure()

for i, company in enumerate(top_companies):
    company_data = normalized_df[normalized_df['회사명'] == company]
    if not company_data.empty:
        values = company_data[radar_metrics_extended].iloc[0].tolist()
        values += [values[0]]  # 첫 번째 값을 마지막에 추가하여 레이더 차트 닫기
        
        fig5.add_trace(go.Scatterpolar(
            r=values,
            theta=radar_labels_extended + [radar_labels_extended[0]],
            fill='toself',
            name=company,
            line_color=colors_radar[i],
            fillcolor=colors_radar[i],
            opacity=0.3 if company != 'LG전자' else 0.5
        ))

fig5.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        )),
    title={
        'text': '🎯 글로벌 Peer Group 종합 경쟁력 비교',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'family': 'Arial Black'}
    },
    height=600,
    showlegend=True
)

fig5.show()

# 종합 점수 계산 (모든 지표의 평균)
comprehensive_scores = normalized_df.groupby('회사명')[radar_metrics_extended].mean().mean(axis=1).sort_values(ascending=False)

print("\n🏆 종합 경쟁력 순위:")
for i, (company, score) in enumerate(comprehensive_scores.items(), 1):
    symbol = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}위"
    highlight = " ⭐" if company == 'LG전자' else ""
    print(f"   {symbol} {company}: {score:.3f}점{highlight}")

# LG전자 세부 분석
lg_rank = list(comprehensive_scores.index).index('LG전자') + 1
lg_score = comprehensive_scores['LG전자']

print(f"\n📊 LG전자 종합 경쟁력 분석:")
print(f"   • 종합 순위: {lg_rank}/{len(df)}위")
print(f"   • 종합 점수: {lg_score:.3f}점")
print(f"   • 상위 비율: {((len(df) - lg_rank + 1) / len(df) * 100):.1f}%")

# 강점/약점 분석
lg_normalized = normalized_df[normalized_df['회사명'] == 'LG전자'][radar_metrics_extended].iloc[0]
strengths = []
weaknesses = []

for metric, label in zip(radar_metrics_extended, radar_labels_extended):
    score = lg_normalized[metric]
    if score >= 0.7:
        strengths.append(f"{label} ({score:.2f})")
    elif score <= 0.3:
        weaknesses.append(f"{label} ({score:.2f})")

print(f"\n💪 LG전자 주요 강점:")
if strengths:
    for strength in strengths:
        print(f"   • {strength}")
else:
    print("   • 뚜렷한 강점 지표 없음 (0.7 이상)")

print(f"\n⚠️ LG전자 개선 필요 분야:")
if weaknesses:
    for weakness in weaknesses:
        print(f"   • {weakness}")
else:
    print("   • 뚜렷한 약점 지표 없음 (0.3 이하)")


📈 5. 종합 경쟁력 분석
----------------------------------------



🏆 종합 경쟁력 순위:
   🥇 Continental: 0.591점
   🥈 LG전자: 0.562점 ⭐
   🥉 Denso: 0.538점
   4위 Haier: 0.444점
   5위 Whirlpool: 0.355점
   6위 Daikin: 0.354점
   7위 Electrolux: 0.343점
   8위 Hisense: 0.319점

📊 LG전자 종합 경쟁력 분석:
   • 종합 순위: 2/8위
   • 종합 점수: 0.562점
   • 상위 비율: 87.5%

💪 LG전자 주요 강점:
   • 밸류에이션(P/E) (0.86)
   • 밸류에이션(P/B) (1.00)
   • 밸류에이션(EV/EBITDA) (1.00)

⚠️ LG전자 개선 필요 분야:
   • 영업이익률 (0.15)
   • ROE (0.05)


## 🎯 전략적 결론 및 투자 관점

### 📊 LG전자 글로벌 경쟁력 종합 평가

#### 🔍 **주요 발견사항**

1. **📈 규모의 우위**
   - 시가총액 4위 (159억 달러) - 중견 글로벌 기업 수준
   - 아시아 기업 중에서는 Denso, Haier 다음으로 3위

2. **💰 밸류에이션 매력도**
   - P/E Ratio 6.9x (2025F) - 업계 대비 저평가
   - P/B Ratio 0.7x - 자산 대비 매우 저평가
   - EV/EBITDA 2.9x - 현금흐름 대비 저평가

3. **📊 수익성 개선 추세**
   - 영업이익률: 4.2% (2023) → 5.0% (2025F)
   - ROE: 3.3% (2023) → 10.7% (2025F) - 극적 개선
   - 매출 성장률: 5.3% (2025F) - 안정적 성장

#### 🏆 **경쟁 포지션 분석**

**강점 영역:**
- **저평가 매력**: 모든 밸류에이션 지표에서 업계 최저 수준
- **수익성 개선**: ROE 3배 이상 급상승 전망
- **안정적 성장**: 지속가능한 매출 성장률

**개선 필요 영역:**
- **절대적 수익성**: 영업이익률이 업계 평균 이하
- **성장성**: 다른 아시아 기업 대비 성장률 낮음

#### 💡 **투자 관점 및 전략 제언**

1. **📈 가치투자 관점**
   - 현재 밸류에이션 수준은 매우 매력적
   - 수익성 개선이 주가에 반영될 가능성 높음

2. **🚀 성장동력 확보 필요**
   - AI/IoT, 전기차 부품 등 신성장 동력 강화
   - 글로벌 시장에서의 프리미엄 포지셔닝

3. **⚡ 운영 효율성 제고**
   - 마진 개선을 통한 수익성 극대화
   - 자본 효율성 지속 개선

### 📋 **요약**
LG전자는 **저평가된 가치주**의 특성을 보이며, 수익성 개선 추세가 뚜렷한 **턴어라운드 스토리**를 가지고 있음. 글로벌 경쟁사 대비 밸류에이션 매력도가 높아 중장기 투자 관점에서 긍정적 평가.